In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc

In [15]:
# Physical parameters
kappa = 1.09e-6
rho_i = 917
c_p = 2009
L_c = 3.34e5

k = kappa * rho_i * c_p

# Initial geometry
H0 = 23.0
dz = 1.0

# Fixed computational grid
M = int(round(H0 / dz))
xi = np.linspace(0, 1, M + 1)
dxi = xi[1] - xi[0]

# Initial and boundary temperatures
T_s, T_b = 0.0, -0.8

# Periodic surface forcing
Delta_Ts = 10.0  # Replace with your chosen amplitude (K)
B = 365.25 * 24 * 3600  # Forcing period (seconds)
omega = 2 * np.pi / B

Q_a = Delta_Ts * np.sqrt(omega * k * rho_i * c_p)

def Q_net(t):
    return Q_a * (1 + np.cos(omega * t))

# Simulation parameters
n_periods = 5
t_final = n_periods * B

r = 0.4
save_every = 10

p = np.poly1d([0.002011, -0.05199, 0.1666, -0.8])

H = H0

z0 = xi * H0

T = p(z0).astype(float)

T[0], T[-1] = T_b, T_s

In [16]:
# Initial snapshots
T_frames = [T.copy()]
H_frames = [H]
t_frames = [0.0]

T_new = np.empty_like(T)

t = 0.0
n = 0

while t < t_final:

    # Physical coordinates at current time
    z = xi * H

    # Surface temperature gradient (second-order backward difference)
    dT_dz_surface = (
        3*T[-1] - 4*T[-2] + T[-3]
    ) / (2 * H * dxi)

    # Moving boundary equation
    H_dot = (
        k * dT_dz_surface - Q_net(t)
    ) / (rho_i * L_c)

    # Adaptive time step for diffusion and advection stability
    dt_diff = r * (H * dxi)**2 / kappa

    a_max = abs(H_dot / H)
    dt_adv = 0.4 * dxi / a_max if a_max > 0 else np.inf

    # Resolve periodic forcing with at least 200 steps per period
    dt_forcing = B / 200

    dt = min(dt_diff, dt_adv, dt_forcing, t_final - t)

    # Forward Euler + centered spatial differences
    T_new[1:-1] = T[1:-1] + dt * (
        (kappa / H**2) *
        (T[2:] - 2*T[1:-1] + T[:-2]) / dxi**2

        + (xi[1:-1] * H_dot / H) *
        (T[2:] - T[:-2]) / (2*dxi)
    )

    # Enforce temperature boundary conditions
    T_new[0] = T_b
    T_new[-1] = T_s

    # Update moving boundary
    H_new = H + dt * H_dot

    if H_new <= 0:
        raise RuntimeError("Ice thickness reached zero.")

    # Advance solution
    T[:] = T_new
    H = H_new
    t += dt
    n += 1

    # Save snapshots
    if n % save_every == 0:
        T_frames.append(T.copy())
        H_frames.append(H)
        t_frames.append(t)

# Save final state
if t_frames[-1] != t:
    T_frames.append(T.copy())
    H_frames.append(H)
    t_frames.append(t)

T_frames = np.array(T_frames)
H_frames = np.array(H_frames)
t_frames = np.array(t_frames)

print(f"Completed {n} time steps")
print(f"Final time: {t/B:.2f} forcing periods")
print(f"Initial thickness: {H0:.3f} m")
print(f"Final thickness: {H:.3f} m")

Completed 1000 time steps
Final time: 5.00 forcing periods
Initial thickness: 23.000 m
Final thickness: 18.764 m


In [17]:
fig, ax = plt.subplots(figsize=(8, 6))
plt.close()

ax.set_ylim(0, np.max(H_frames))
ax.set_xlim(np.min(T_frames), np.max(T_frames))

ax.set_ylabel("Depth (m)")
ax.set_xlabel("Temperature")
ax.set_title("Moving-Boundary Heat Equation")

line, = ax.plot([], [], lw=2)

def init():
    line.set_data([], [])
    return (line,)

def animate(j):

    # Reconstruct physical spatial coordinates
    z_j = xi * H_frames[j]

    # Temperature at this time step
    T_j = T_frames[j, :]

    line.set_data(T_j, z_j)

    time_years = t_frames[j] / (365.25 * 24 * 3600)

    ax.set_title(
        f"Temperature Evolution: "
        f"t = {time_years:.2f} years, "
        f"H = {H_frames[j]:.2f} m"
    )

    return (line,)

anim = animation.FuncAnimation(
    fig,
    animate,
    init_func=init,
    frames=len(T_frames),
    interval=50,
    blit=True
)

rc('animation', html='jshtml')
anim